# 5.1. 层和块

块（block）可以描述单个层、由多个层组成的组件或整个模型本身。请自己完成每个代码单元。

导入所需模块：torch、torch.nn、torch.nn.functional（F）。

In [ ]:
import torch
from torch import torch.nn 


用 nn.Sequential 定义一个多层感知机 net：nn.Linear(20, 256) → nn.ReLU() → nn.Linear(256, 10)；生成 X = torch.rand(2, 20) 并调用 net(X)。

### 5.1.1. 自定义块

继承 nn.Module 编写一个 MLP 类：
- __init__ 中调用 super().__init__()，声明 self.hidden = nn.Linear(20, 256)（隐藏层）、self.out = nn.Linear(256, 10)（输出层）
- forward(self, X)：return self.out(F.relu(self.hidden(X)))

实例化 net = MLP() 并用 X 调用。

### 5.1.2. 顺序块

自己实现一个简化的 MySequential，与 nn.Sequential 行为一致：
- __init__(self, *args)：调用 super().__init__()，用 enumerate 遍历 args，把每个模块存入 self._modules[str(idx)]
- forward(self, X)：for block in self._modules.values(): X = block(X)，最后 return X

实例化并测试。

### 5.1.3. 在前向传播函数中执行代码

块可以在前向传播中做更灵活的事。实现 FixedHiddenMLP：
- __init__ 声明 self.rand_weight = torch.rand((20, 20), requires_grad=False)（不参与反向传播）和 self.linear = nn.Linear(20, 20)
- forward(self, X)：X = self.linear(X)；X = F.relu(X @ self.rand_weight + 1)；X = self.linear(X)；再用 while X.abs().sum() > 1: X /= 2 调节，最后 return X.sum()

还可把自定义块嵌进 nn.Sequential（例如精妙混合使用）。实例化并用 X 测试。

### 5.1.4. 效率

（了解）框架尽力避免不必要地评估每个项，构建计算图时框架会跳过无关的项。

### 5.1.5. 小结

- 块可以组合层形成递归结构，简化复杂网络的实现
- 自定义块需要继承 nn.Module 并实现 forward 前向传播函数
- 块内部可以自由执行代码，如调用其他块、控制流、数组运算等

### 5.1.6. 练习

1. 如果 MLP 改为不加 ReLU 而是直接输出，会发生什么？
2. 尝试自定义一个由块构成的、包含若干层和参数的结构。
3. 若 forward 中用到其他 Python 控制流程（if、for 等），梯度计算是否仍正确？